In [1]:

import nibabel as nib
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path(r"D:\PS07_CT2MRI")
BASE = ROOT / "dataset" / "volumes"

manifest = pd.read_csv(ROOT / "dataset" / "manifest.csv")

results = []

for pid in manifest["id"]:

    folder = BASE / pid

    ct = np.asarray(
        nib.load(str(folder / "ct.nii.gz")).dataobj
    )
    mr = np.asarray(
        nib.load(str(folder / "mr.nii.gz")).dataobj
    )
    mask = np.asarray(
        nib.load(str(folder / "mask.nii.gz")).dataobj
    )

    results.append({
        "patient": pid,
        "ct_finite": np.isfinite(ct).all(),
        "mr_finite": np.isfinite(mr).all(),
        "mask_finite": np.isfinite(mask).all(),
        "mask_nonempty": np.any(mask > 0),
        "mask_values": set(np.unique(mask).tolist()).issubset({0, 1}),
        "ct_shape": ct.shape == (192, 224, 192),
        "mr_shape": mr.shape == (192, 224, 192),
        "mask_shape": mask.shape == (192, 224, 192)
    })

df = pd.DataFrame(results)

checks = [
    "ct_finite", "mr_finite", "mask_finite",
    "mask_nonempty", "mask_values",
    "ct_shape", "mr_shape", "mask_shape"
]

print("Total patients:", len(df))

for check in checks:
    print(f"{check}: {df[check].sum()} / {len(df)}")

failed = df[~df[checks].all(axis=1)]

print("\nPatients with data issues:")
print(failed[["patient"] + checks].to_string(index=False))

df.to_csv(ROOT / "results" / "dataset_audit.csv", index=False)

print("\nAudit saved to results/dataset_audit.csv")


Total patients: 120
ct_finite: 120 / 120
mr_finite: 120 / 120
mask_finite: 120 / 120
mask_nonempty: 120 / 120
mask_values: 120 / 120
ct_shape: 120 / 120
mr_shape: 120 / 120
mask_shape: 120 / 120

Patients with data issues:
Empty DataFrame
Columns: [patient, ct_finite, mr_finite, mask_finite, mask_nonempty, mask_values, ct_shape, mr_shape, mask_shape]
Index: []

Audit saved to results/dataset_audit.csv
